# สำรวจข้อมูล 1/4: สถิติเบื้องต้น (Descriptive Statistics)

| ไฟล์ | เกณฑ์ 3.5.2 (18 คะแนน) |
|---|---|
| **→** eda_step1_descriptive | 3.5.2.1 สถิติเบื้องต้น (8) |
|  eda_step2_trends | 3.5.2.2 แนวโน้ม/ความสัมพันธ์ (5) |
|  eda_step3_outliers_missing | 3.5.2.3 ค่าผิดปกติ/การขาดหาย (3) |
|  eda_step4_insights | 3.5.2.4 Insight + ตอบคำถามจาก step1_questions (2) |

ต่อจากการทำความสะอาด 5 ขั้น ตอนนี้ข้อมูลพร้อมแล้ว ขั้นนี้คือ **ทำความรู้จักข้อมูลด้วยตัวเลข** ก่อนหาแนวโน้มหรือข้อสรุป

ตอบเกณฑ์ 3.5.2.1 (8 คะแนน) ครบ 4 ข้อ:
1. ภาพรวมชุดข้อมูล: shape และชนิดตัวแปร
2. สถิติสรุป: mean / median / std / IQR / min–max
3. การกระจาย: histogram และ boxplot (อย่างน้อย 2 ตัวแปร)
4. สัดส่วนหมวดหมู่

**อินพุต:** `final/energy_clean.csv` · **เอาต์พุต:** รูปและตารางในโฟลเดอร์ `eda_output/` (ใช้ทำสไลด์)

> วางไฟล์นี้ในโฟลเดอร์ `energy_project` แล้วรัน **Kernel → Restart Kernel and Run All Cells**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DATA = os.path.join("final", "energy_clean.csv")
if not os.path.exists(DATA):
    raise FileNotFoundError("ไม่พบ final/energy_clean.csv ให้รันขั้นทำความสะอาด 1-5 ให้สำเร็จก่อน")
OUT = "eda_output"                       # เก็บรูปและตารางไว้ใช้ทำสไลด์
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(DATA)
LATEST = int(df.loc[df["year_complete"], "year"].max())   # ปีล่าสุดที่ข้อมูลครบ = 2021
y = df[df["year"] == LATEST].copy()                        # ภาพตัดขวาง (ทุกประเทศ ปีเดียว)
ASEAN = sorted(df.loc[df["is_asean"], "country"].unique())
FUEL_COLOR = {"coal": "#4A4039", "oil": "#8C5A3C", "gas": "#E0913A", "nuclear": "#7B61A8", "hydro": "#2E78AD",
              "wind": "#5FB3A1", "solar": "#E8B923", "biofuel": "#7A9A3A", "other_renewables": "#9AA5A8"}

def savefig(name):
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")

print(f"ข้อมูล: {df.shape[0]:,} แถว x {df.shape[1]} คอลัมน์ | {df['country'].nunique()} ประเทศ | ปี {df['year'].min()}-{df['year'].max()}")
print(f"ปีที่ใช้วิเคราะห์ภาพตัดขวาง: {LATEST} ({len(y)} ประเทศ)")

## 1.1 ภาพรวมชุดข้อมูล: ตัวแปรมีกี่ชนิด
แบ่งตัวแปรเป็น 4 ชนิด เพราะแต่ละชนิดใช้สถิติและกราฟต่างกัน
- **ตัวเลข (numeric)** → mean, median, histogram
- **หมวดหมู่ (categorical)** → นับจำนวน สัดส่วน, bar chart
- **เวลา (time)** → แนวโน้ม, line chart
- **ธง (boolean)** → สัดส่วน True/False

In [ ]:
def kind(c):
    if c == "year": return "เวลา (time)"
    if pd.api.types.is_bool_dtype(df[c]): return "ธง (boolean)"
    if pd.api.types.is_numeric_dtype(df[c]): return "ตัวเลข (numeric)"
    return "หมวดหมู่ (categorical)"

types = pd.DataFrame({"column": df.columns, "ชนิด": [kind(c) for c in df.columns]})
summary = types.groupby("ชนิด")["column"].agg(["count", lambda s: ", ".join(s[:6]) + (" ..." if len(s) > 6 else "")])
summary.columns = ["จำนวนคอลัมน์", "ตัวอย่าง"]
summary

In [ ]:
# ตัวแปรสำคัญที่ตอบคำถามของเรา (ใช้ตลอดการสำรวจ)
KEY = {
    "per_capita_electricity": "ไฟฟ้าต่อคน (kWh)",
    "renewables_share_elec": "สัดส่วนหมุนเวียน (%)",
    "fossil_share_elec": "สัดส่วนฟอสซิล (%)",
    "gas_share_elec": "สัดส่วนก๊าซ (%)",
    "carbon_intensity_elec": "ความเข้มคาร์บอน (gCO2/kWh)",
    "electricity_generation": "ไฟฟ้าที่ผลิต (TWh)",
}
print("ตัวแปรสำคัญ:", len(KEY), "ตัว | หน่วยวิเคราะห์: 1 ประเทศ ใน 1 ปี")

## 1.2 สถิติสรุปของตัวแปรสำคัญ
ใช้ข้อมูล **ปี 2021 (ปีล่าสุดที่ครบ)** เพื่อเทียบประเทศในเวลาเดียวกัน ถ้ารวมทุกปี ประเทศเดียวจะถูกนับ 22–23 ครั้ง

| สถิติ | บอกอะไร |
|---|---|
| mean, median | ค่ากลาง (ถ้าต่างกันมาก = ข้อมูลเบ้) |
| std, IQR | การกระจาย (IQR = Q3 − Q1 ไม่ถูกค่าสุดโต่งดึง) |
| min, max | ขอบเขตจริงของข้อมูล |
| skew | ความเบ้ (> 1 = เบ้ขวามาก) |

In [ ]:
def describe(frame, cols):
    s = pd.DataFrame({
        "n": frame[cols].count(),
        "mean": frame[cols].mean(),
        "median": frame[cols].median(),
        "std": frame[cols].std(),
        "Q1": frame[cols].quantile(.25),
        "Q3": frame[cols].quantile(.75),
        "min": frame[cols].min(),
        "max": frame[cols].max(),
        "skew": frame[cols].skew(),
    })
    s["IQR"] = s["Q3"] - s["Q1"]
    s["mean/median"] = s["mean"] / s["median"]
    s.index = [KEY.get(c, c) for c in cols]
    return s[["n", "mean", "median", "std", "IQR", "min", "max", "skew", "mean/median"]].round(1)

stats = describe(y, list(KEY))
stats.to_csv(os.path.join(OUT, "stats_2021.csv"), encoding="utf-8-sig")
stats

**อ่านตาราง (ใช้ตัวเลขจากตารางด้านบน):**
- **ไฟฟ้าต่อคน**: mean ≈ 3,985 แต่ median ≈ 2,325 kWh (mean สูงกว่า 1.7 เท่า) และ skew ≈ 4.3 → **เบ้ขวามาก** มีไม่กี่ประเทศที่ใช้ไฟสูงมาก ดังนั้นต้องรายงาน **median** เป็นค่ากลาง
- **ไฟฟ้าที่ผลิต**: skew > 10 เพราะจีน สหรัฐฯ ใหญ่กว่าประเทศอื่นหลายร้อยเท่า → ไม่เหมาะใช้เทียบประเทศ ให้ใช้ค่า **ต่อหัว** หรือ **สัดส่วน %** แทน
- **สัดส่วนก๊าซ**: median แค่ 2.9% แต่ mean 20% → ประเทศส่วนใหญ่แทบไม่ใช้ก๊าซ มีกลุ่มเล็กที่ใช้มาก (ไทยอยู่ในกลุ่มนี้)
- **สัดส่วนหมุนเวียน**: กระจายกว้างมาก (0–100%, std ≈ 32) → แต่ละประเทศต่างกันสุดขั้ว

In [ ]:
# ประเทศไทยอยู่ตรงไหนของการกระจาย (เปอร์เซ็นไทล์)
th = y[y["country"] == "Thailand"].iloc[0]
pos = pd.DataFrame({
    "ไทย": [th[c] for c in KEY],
    "มัธยฐานโลก": [y[c].median() for c in KEY],
    "เปอร์เซ็นไทล์ของไทย": [round((y[c].dropna() < th[c]).mean() * 100) for c in KEY],
}, index=list(KEY.values())).round(1)
pos

> **อ่านผล:** เปอร์เซ็นไทล์ 86 แปลว่าไทยสูงกว่า 86% ของประเทศทั้งหมด
> - ไทยใช้ **ก๊าซ 64.5%** สูงกว่า 86% ของโลก (มัธยฐานโลกแค่ 2.9%)
> - สัดส่วน **หมุนเวียน 14.5%** ต่ำกว่ามัธยฐานโลก (25.2%) อยู่ที่เปอร์เซ็นไทล์ 35
> - ไฟฟ้าต่อคนของไทยอยู่ **กลางโลกพอดี** (เปอร์เซ็นไทล์ 51)

## 1.3 การกระจาย: Histogram และ Boxplot
- **Histogram** เห็นรูปร่าง (สมมาตร / เบ้ / มีสองยอด)
- **Boxplot** เห็นค่ากลาง ช่วง IQR และค่าที่อยู่ไกลกลุ่ม

ไฟฟ้าต่อคนเบ้มาก จึงแสดงทั้ง **สเกลปกติ** และ **สเกล log** (log ทำให้เห็นรายละเอียดของประเทศส่วนใหญ่ที่ใช้ไฟน้อย)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
pc = y["per_capita_electricity"].dropna()
ax[0, 0].hist(pc, bins=40, color="#1F5F8B"); ax[0, 0].set_title("Electricity per person (linear)")
ax[0, 0].axvline(pc.median(), color="red", ls="--", label=f"median {pc.median():,.0f}"); ax[0, 0].axvline(pc.mean(), color="orange", ls=":", label=f"mean {pc.mean():,.0f}")
ax[0, 0].legend(); ax[0, 0].set_xlabel("kWh per person")
ax[0, 1].hist(np.log10(pc[pc > 0]), bins=30, color="#1F5F8B"); ax[0, 1].set_title("Electricity per person (log10)")
ax[0, 1].set_xlabel("log10(kWh per person)  e.g. 3 = 1,000 kWh")
ax[0, 2].boxplot(np.log10(pc[pc > 0]), vert=False); ax[0, 2].set_title("Boxplot (log10)")

rs = y["renewables_share_elec"].dropna()
ax[1, 0].hist(rs, bins=20, color="#5FB3A1"); ax[1, 0].set_title("Renewables share of electricity")
ax[1, 0].axvline(rs.median(), color="red", ls="--", label=f"median {rs.median():.1f}%"); ax[1, 0].legend(); ax[1, 0].set_xlabel("%")
gs = y["gas_share_elec"].dropna()
ax[1, 1].hist(gs, bins=20, color="#E0913A"); ax[1, 1].set_title("Gas share of electricity")
ax[1, 1].axvline(th["gas_share_elec"], color="#1F5F8B", lw=2.5, label=f"Thailand {th['gas_share_elec']:.1f}%"); ax[1, 1].legend(); ax[1, 1].set_xlabel("%")
ax[1, 2].boxplot([rs, gs, y["fossil_share_elec"].dropna()]); ax[1, 2].set_xticks([1, 2, 3], ["renewables", "gas", "fossil"])
ax[1, 2].set_title("Shares (%) boxplot")
plt.suptitle(f"Distributions across countries, {LATEST}", fontsize=14); plt.tight_layout()
savefig("dist_key_vars.png"); plt.show()

**เห็นอะไร → หมายความว่า → ทำอะไรต่อ**
- **ไฟฟ้าต่อคน (สเกลปกติ)** กองอยู่ฝั่งซ้าย หางยาวไปขวา แต่ **สเกล log** กระจายสมดุลขึ้นมาก (เบ้ซ้ายเล็กน้อยจากประเทศที่ใช้ไฟน้อยมาก) → เวลาแสดงบนแดชบอร์ดหรือหาความสัมพันธ์ ควรใช้ **สเกล log**
- **สัดส่วนหมุนเวียน** มีสองกลุ่มชัด: กลุ่มใกล้ 0% (พึ่งฟอสซิลเกือบหมด) และกลุ่มใกล้ 100% (มีเขื่อนใหญ่) → การรายงาน "ค่าเฉลี่ยโลก" อย่างเดียวจะซ่อนความจริงข้อนี้
- **สัดส่วนก๊าซ** ประเทศส่วนใหญ่อยู่ใกล้ 0% แต่ไทย (เส้นน้ำเงิน) อยู่ไกลทางขวา → ประเด็นนี้เป็นจุดเด่นที่ควรนำเสนอ

## 1.4 สัดส่วนหมวดหมู่
ข้อมูลนี้ไม่มี "label" สำหรับจำแนก แต่มีหมวดหมู่ที่ต้องดูความสมดุล:
1. **ทวีป**: ถ้าทวีปหนึ่งมีประเทศมาก ค่ารวมทั้งโลกจะเอนเอียงไปทางนั้น
2. **เชื้อเพลิงหลักของแต่ละประเทศ**: สร้างหมวดใหม่จากข้อมูล (feature) = เชื้อเพลิงที่มีสัดส่วนสูงสุด
3. **ธงคุณภาพข้อมูล**: สัดส่วนแถวที่ต้องระวัง

In [ ]:
FUEL_SHARE = {"coal": "coal_share_elec", "oil": "oil_share_elec", "gas": "gas_share_elec", "nuclear": "nuclear_share_elec",
              "hydro": "hydro_share_elec", "wind": "wind_share_elec", "solar": "solar_share_elec",
              "biofuel": "biofuel_share_elec", "other_renewables": "other_renewables_share_elec_exc_biofuel"}
has_mix = y[list(FUEL_SHARE.values())].notna().any(axis=1)          # ตัดประเทศที่ไม่มีข้อมูลไฟฟ้าปีนี้
y.loc[has_mix, "main_fuel"] = y.loc[has_mix, list(FUEL_SHARE.values())].idxmax(axis=1).map({v: k for k, v in FUEL_SHARE.items()})

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
reg = y["region"].value_counts()
ax[0].barh(reg.index[::-1], reg.values[::-1], color="#9AA5A8")
for i, v in enumerate(reg.values[::-1]): ax[0].text(v + 0.5, i, f"{v} ({v / reg.sum() * 100:.0f}%)", va="center")
ax[0].set_title("Countries per region"); ax[0].set_xlim(0, reg.max() * 1.25)
mf = y["main_fuel"].value_counts()
ax[1].barh(mf.index[::-1], mf.values[::-1], color=[FUEL_COLOR[f] for f in mf.index[::-1]])
for i, v in enumerate(mf.values[::-1]): ax[1].text(v + 0.5, i, f"{v} ({v / mf.sum() * 100:.0f}%)", va="center")
ax[1].set_title(f"Main fuel for electricity (number of countries, {LATEST})"); ax[1].set_xlim(0, mf.max() * 1.25)
plt.tight_layout(); savefig("category_balance.png"); plt.show()

print("ไทยใช้เชื้อเพลิงหลัก:", y.loc[y["country"] == "Thailand", "main_fuel"].iloc[0])
print("ประเทศที่ใช้ก๊าซเป็นเชื้อเพลิงหลัก:", mf.get("gas", 0), "จาก", mf.sum(), "ประเทศ")

In [ ]:
flags = pd.DataFrame({
    "สัดส่วน True (%)": [round(df[c].mean() * 100, 1) for c in ["year_complete", "gdp_available", "is_outlier_pc_elec", "low_precision", "is_asean"]],
}, index=["ปีข้อมูลครบ", "มี GDP", "ไฟฟ้าต่อคนสูงผิดปกติ (จริง)", "ตัวเลขถูกปัด (ประเทศเล็ก)", "อาเซียน"])
flags

**อ่านผล:**
- **แอฟริกาและเอเชีย** มีประเทศมากที่สุด (รวมเกือบครึ่ง) ค่ามัธยฐานโลกจึงสะท้อนประเทศกำลังพัฒนาเป็นหลัก ข้อนี้ต้องระบุเป็นข้อจำกัด
- **น้ำมันเป็นเชื้อเพลิงหลักของประเทศมากที่สุด** แต่ส่วนใหญ่เป็นประเทศเกาะขนาดเล็ก รองลงมาคือ **พลังน้ำ** ส่วน **ก๊าซ** (แบบไทย) อยู่อันดับ 3
- อาเซียนมีแค่ราว 5% ของแถวทั้งหมด → เวลาเทียบไทยกับอาเซียน กลุ่มเปรียบเทียบเล็ก (10 ประเทศ) ต้องระวังการสรุปเกินหลักฐาน

## ✅ สรุปขั้นนี้ (เกณฑ์ 3.5.2.1)
| ข้อย่อย | หลักฐาน |
|---|---|
| 3.5.2.1.1 ภาพรวม shape/ชนิดตัวแปร | ข้อ 1.1 |
| 3.5.2.1.2 สถิติสรุป mean/median/std/IQR/min–max | ข้อ 1.2 + `eda_output/stats_2021.csv` |
| 3.5.2.1.3 histogram/boxplot ≥ 2 ตัวแปร | ข้อ 1.3 + `eda_output/dist_key_vars.png` (3 ตัวแปร) |
| 3.5.2.1.4 สัดส่วนหมวดหมู่ | ข้อ 1.4 + `eda_output/category_balance.png` |

**ข้อค้นพบสำคัญที่ใช้ต่อ:** ข้อมูลเบ้มาก ต้องใช้ median และสเกล log · ไทยใช้ก๊าซสูงกว่าเกือบทุกประเทศ · สัดส่วนหมุนเวียนแบ่งเป็นสองกลุ่มชัด

**ไฟล์ถัดไป:** `eda_step2_trends.ipynb` แนวโน้มตามเวลา และความสัมพันธ์ระหว่างตัวแปร